# Correction Training — real ImageNet sampling (Colab, GPU)

Official code: [sunovivid/correction-training](https://github.com/sunovivid/correction-training) (MIT) · Paper: [arXiv:2610.14136](https://arxiv.org/abs/2610.14136) · Project: [correction-training.github.io](https://correction-training.github.io/)

> ⚠️ **GPU budget needs owner approval.** This notebook is prepared but **not executed**. Sampling requires an **NVIDIA GPU** (a free Colab T4 works for small grids, but is slow for SiT-XL) and downloads **~11 GB per SiT/SiT-REPA checkpoint**. The free Colab tier may still work; anything paid (A100/L4 runtime, Colab Pro) should be explicitly approved by the repo owner first. The CPU 2D toy in `apps/correction-training-2d/` never needs any of this.

What you get: 16 golden retrievers (class 207) from **SiT-REPA-XL/2** sampled **with K>0 iterative corrections and no CFG**, plus a K-sweep so you can see the correction effect on real images.

In [ ]:
# 0) Check the GPU Colab gave you (Runtime > Change runtime type > GPU)
!nvidia-smi

In [ ]:
# 1) Clone the official repo and install deps (Colab already ships Python 3.11+ and CUDA torch)
!git clone https://github.com/sunovivid/correction-training.git
%cd correction-training
!python -m pip install -q -r requirements-latent.txt

## 2) Sample SiT-REPA-XL/2 with correction (K > 0)

First run auto-downloads the checkpoint from the official
[release](https://github.com/sunovivid/correction-training/releases/tag/imagenet-checkpoints-v1)
— **about 11 GB**, so make sure the Colab disk has room and you are OK with the bandwidth.
If the download crawls below 1 MB/s, interrupt and rerun (upstream README tip).

In [ ]:
# 16 golden retrievers, default K from the config, no CFG
!python src/sample.py --config configs/eval/qualitative/sit-repa.yml --output samples/sit-repa

In [ ]:
# K-sweep on the same seed/class: K=0 baseline vs K=4 vs K=8 corrections per step
for K in [0, 4, 8]:
    !python src/sample.py --config configs/eval/qualitative/sit-repa.yml \
        --corrections_per_step {K} --samples 8 --batch-size 4 --seed 0 \
        --classes 207 --output samples/k-sweep-K{K}

In [ ]:
# 3) Display the newest grid from each run
import glob
from IPython.display import Image, display
for d in sorted(glob.glob('samples/*/')):
    pngs = sorted(glob.glob(d + '**/*.png', recursive=True))
    if pngs:
        print(d)
        display(Image(pngs[-1], width=512))

## Notes
- `--corrections_per_step` is **K in the paper** (Algorithm 2); `--cfg` other than 1 requires K=0.
- `--correction-interval start end` restricts correction to a t-range (1 = noise, 0 = data).
- Paper-setting NFE-matched configs are documented in Appendix B.1 and `docs/evaluation.md` upstream.
- JiT-L/16 (`configs/eval/qualitative/jit.yml`) is lighter if disk is tight.
- Set `CORRECTION_CHECKPOINT_DIR` to keep checkpoints on Drive between sessions.